# Knowledge Distillation — ResNet-50 to a Compact CNN

Train a 288,746-parameter CNN twice on the same images, with the same
architecture, seed, optimiser and schedule. Change exactly one thing: whether it is asked
to match the **hard one-hot label** or the **soft class distribution** a fine-tuned
ResNet-50 produces.

| Model | Parameters | Test accuracy |
|---|---:|---:|
| ResNet-50 teacher | 23,528,522 | 90.03% |
| Student CNN — baseline | 288,746 | 83.26% |
| Student CNN — distilled (T=4, α=0.7) | 288,746 | 83.30% |

**+0.04 points.** The teacher is real — 90.0%, 6.8 points clear of the student,
81× larger. The gain is not. Three experiments establish that, and the two negative
ones are the interesting part.

Everything is in this notebook: the models, the training loops, the α sweep, and the
measured numbers. Section 7 has the results as recorded in `results.json`; sections 1–5 are
the code that produced them, runnable end to end on a CPU in about three hours.

**Accuracy convention throughout:** full 10,000-image test set, each run's *final*
weights. Not the best epoch — that would be a peek at the test set.

## 1. The objective

Hinton, Vinyals & Dean (2015) train the student against the teacher's softened output
rather than the label:

$$\mathcal{L} = \alpha \cdot T^2 \cdot \mathrm{KL}\!\left(\sigma(z_s/T) \,\|\, \sigma(z_t/T)\right) + (1-\alpha)\cdot \mathrm{CE}(z_s, y)$$

Two details in that line are easy to skip and both matter.

**Why soft targets carry more than labels.** A one-hot label says an image is a cat. The
teacher says 0.82 cat, 0.11 dog, 0.005 airplane. The relative weight on the *wrong* classes
is a statement about which classes resemble each other, and the student learns that
structure from every image — not only from the ones it gets wrong.

**Why the $T^2$.** Dividing logits by $T$ shrinks that term's gradient by roughly $1/T^2$.
Without the correction, raising the temperature would quietly lower the learning rate on it
too, and $\alpha$ would stop meaning the same thing from one temperature to the next.

Two implementation details in the code below are the usual places this goes wrong:

* `F.kl_div` expects its **first argument to be log-probabilities**. Passing probabilities
  there computes something that is not a KL divergence at all.
* `T = 1` is not distillation. At $T=1$ the teacher's output is nearly one-hot and says
  little the label did not already say.

In [ ]:
import torch
import torch.nn.functional as F


def distillation_loss(student_logits, teacher_logits, targets, T=4.0, alpha=0.7):
    """The whole method. Both arguments are raw logits, never probabilities."""
    soft = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),   # log-probabilities -- required
        F.softmax(teacher_logits / T, dim=1),       # probabilities
        reduction="batchmean",
    ) * (T ** 2)                                    # undo the 1/T^2 gradient shrink
    hard = F.cross_entropy(student_logits, targets) # expects logits, applies log_softmax itself
    return alpha * soft + (1 - alpha) * hard

### What the temperature actually does

Nothing is trained here — just the softmax at three temperatures on one plausible set of
teacher logits, to make "softer" concrete.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})

classes = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
logits = torch.tensor([[1.1, 0.4, 2.2, 6.8, 1.6, 5.1, 0.9, 1.4, 0.7, 0.3]])  # a "cat"

fig, ax = plt.subplots(figsize=(8, 3.2))
for T, colour in [(1.0, "#94a3b8"), (4.0, "#f59e0b"), (10.0, "#e11d48")]:
    ax.plot(np.arange(10), F.softmax(logits / T, dim=1).numpy().ravel(),
            "o-", ms=4, color=colour, label=f"T = {T:g}")
ax.set_xticks(np.arange(10)); ax.set_xticklabels(classes, rotation=30)
ax.set_ylabel("probability"); ax.legend(frameon=False)
ax.set_title("The same teacher logits at three temperatures")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

At $T=1$ the teacher is almost one-hot. At $T=4$ the ranking of the wrong classes — dog
above deer above bird above plane — becomes usable signal. At $T=10$ it flattens towards
uniform and that ranking washes out. $T=4$ throughout.

## 2. The two models

**Teacher — ResNet-50 with its stem left alone.** The standard CIFAR adaptation replaces
the 7×7 stride-2 stem and drops the max-pool, so a 32×32 image is not immediately reduced
to 8×8. Tried first, it was a mistake: that replacement stem is **randomly initialised**,
so every pretrained block downstream receives features it has never seen. That teacher
reached 82.1% — *below* the 83.9% of the student it was meant to teach — and distilling
from it **cost 3.0 points**. Experiment 1 in section 7.

Keeping the stem and upsampling CIFAR to 64×64 lets every pretrained weight do the
job it was trained for. It is also ~2.8× cheaper per step, since `layer1` then runs at
16×16 rather than 32×32. Teacher: 82.1% → **90.0%**.

**Student — three conv blocks.** 288,746 parameters, 1.2% of the teacher's
23,528,522. It stays on native 32×32 — it is the model you would actually deploy.

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50

TEACHER_RES = 64   # the pretrained stem expects to downsample; 32x32 is too small for it


def build_teacher():
    model = resnet50(weights="IMAGENET1K_V1")   # stem untouched
    model.fc = nn.Linear(2048, 10)              # logits, no softmax -- the loss applies it
    for name, param in model.named_parameters():
        if name.startswith(("layer1", "layer2")):
            param.requires_grad = False         # low-level filters transfer as they are
    return model


class StudentCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),      # global pooling, not a big flatten
            nn.Dropout(0.1), nn.Linear(128, n_classes)) # logits, no softmax

    def forward(self, x):
        return self.classifier(self.features(x))


print(f"student: {sum(p.numel() for p in StudentCNN().parameters()):,} parameters")

## 3. Data, and what makes the comparison a comparison

A fixed random **20,000-image subset** of CIFAR-10's 50,000 training images, drawn once
with a seeded generator and reused by every run. That is a CPU budget, not a design choice,
and it is worth saying plainly: distillation helps more when data is scarce, so if anything
this setup flatters the method.

The teacher sees those images at 64×64; the students see them at native 32×32.

Everything about the two student runs is held fixed except the loss: same class, same seed,
same initialisation, same augmentation, same optimiser and 20-epoch cosine schedule, same
images in the same order.

In [ ]:
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

N_TRAIN = 20000
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)


def subset_indices(n_total=50000, n=N_TRAIN, seed=0):
    return np.sort(np.random.default_rng(seed).choice(n_total, size=n, replace=False))


def _tf(train, resize=None):
    ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()] if train else []
    if resize:
        ops += [transforms.Resize(resize, antialias=True)]
    return transforms.Compose(ops + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def loaders(batch_size=48, root="data"):
    idx = subset_indices()
    C = lambda tf, train=True: torchvision.datasets.CIFAR10(root, train=train, download=True, transform=tf)
    L = lambda ds, sh=False: DataLoader(ds, batch_size, shuffle=sh, num_workers=0)
    test, test_t = C(_tf(False), False), C(_tf(False, TEACHER_RES), False)
    return {
        "train":   L(Subset(C(_tf(True)), idx), True),                    # student view
        "train_t": L(Subset(C(_tf(True, TEACHER_RES)), idx), True),       # teacher view
        "plain_t": L(Subset(C(_tf(False, TEACHER_RES)), idx)),            # for the logit cache
        "test": L(test), "test_t": L(test_t),
        "quick":   L(Subset(test, list(range(2000)))),                    # per-epoch curve only
        "quick_t": L(Subset(test_t, list(range(2000)))),
    }

## 4. Training

**Teacher logits are cached** in a single forward pass over the training images before the
student runs, so distillation costs no more per epoch than baseline training. The teacher
scores the *un-augmented* image, which keeps the target stable across epochs — and which
section 8 argues is also this implementation's main weakness.

In [ ]:
import time


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


def train_teacher(train_loader, eval_loader, epochs=8, lr=1e-3):
    model = build_teacher()
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for epoch in range(1, epochs + 1):
        model.train(); started = time.time(); total = seen = 0
        for x, y in train_loader:
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); seen += 1
        sched.step()
        print(f"teacher epoch {epoch}: loss {total/seen:.4f}  acc {accuracy(model, eval_loader):.4f}"
              f"  [{(time.time()-started)/60:.1f} min]")
    return model


@torch.no_grad()
def cache_teacher_logits(model, plain_loader):
    """One pass over the training set, in the loader's fixed order."""
    model.eval()
    return torch.cat([model(x) for x, _ in plain_loader])

In [ ]:
def train_student(train_loader, eval_loader, epochs=20, lr=2e-3,
                  distil=False, temperature=4.0, alpha=0.7, label=""):
    """distil=False trains on hard labels only. Everything else is identical."""
    torch.manual_seed(0)                       # same initialisation for every run
    model = StudentCNN()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    history = []

    for epoch in range(1, epochs + 1):
        model.train(); total = seen = 0
        for x, y, *rest in train_loader:
            logits = model(x)
            loss = (distillation_loss(logits, rest[0], y, temperature, alpha)
                    if distil else F.cross_entropy(logits, y))
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); seen += 1
        sched.step()
        acc = accuracy(model, eval_loader)
        history.append({"epoch": epoch, "loss": total / seen, "test_acc": acc})
        print(f"{label} epoch {epoch:2d}: loss {total/seen:.4f}  acc {acc:.4f}")
    return model, history, acc


def with_logits(base, logits, batch_size=48):
    """Wraps a dataset so each item also carries its cached teacher logits."""
    class WithLogits(torch.utils.data.Dataset):
        def __len__(self):  return len(base)
        def __getitem__(self, i):
            x, y = base[i]
            return x, y, logits[i]      # same index, same order as the cache
    return DataLoader(WithLogits(), batch_size, shuffle=True, num_workers=0)

The full pipeline. On two CPU cores this is about 50 minutes for the teacher and 17 per
student.

In [ ]:
def run_all():
    L = loaders()
    teacher = train_teacher(L["train_t"], L["quick_t"], epochs=8)
    logits = cache_teacher_logits(teacher, L["plain_t"])

    train_plain = torchvision.datasets.CIFAR10("data", train=True, transform=_tf(True))
    train_subset = Subset(train_plain, subset_indices())
    paired = with_logits(train_subset, logits)

    base, base_hist, _ = train_student(paired, L["quick"], distil=False, label="baseline")
    kd,   kd_hist,   _ = train_student(paired, L["quick"], distil=True,
                                       temperature=4.0, alpha=0.7, label="distilled")

    return {                                   # full test set, final weights
        "teacher_acc":   accuracy(teacher, L["test_t"]),
        "baseline_acc":  accuracy(base, L["test"]),
        "distilled_acc": accuracy(kd, L["test"]),
        "baseline_history": base_hist, "distilled_history": kd_hist,
    }

# results = run_all()   # ~3 hours on CPU; the recorded output is loaded in section 7

## 5. The α sweep, and how to run one honestly

α = 0.7 puts most of the weight on matching a distribution a 288,746-parameter
network can only partly represent. Worth testing rather than assuming — but a sweep is only
worth anything with a selection protocol:

* **2,000 of the 20,000 training images are held out as validation.** Nothing trains on them.
* every run — baseline and each α — trains on the same 18,000 images, same seed, same schedule.
* **α is chosen on validation accuracy**, and only the chosen model is then scored on test.

Choosing α by test accuracy and reporting that number is how a sweep becomes a lie: with
enough values you will always find one that looks good on the test set.

In [ ]:
def sweep(alphas=(0.3, 0.5, 0.7, 0.9), n_val=2000):
    idx = subset_indices()
    order = np.random.default_rng(1).permutation(len(idx))
    val_pos, train_pos = np.sort(order[:n_val]), np.sort(order[n_val:])

    L = loaders()
    teacher = train_teacher(L["train_t"], L["quick_t"], epochs=8)
    logits = cache_teacher_logits(teacher, L["plain_t"])   # indexed by subset position

    aug = torchvision.datasets.CIFAR10("data", train=True, transform=_tf(True))
    plain = torchvision.datasets.CIFAR10("data", train=True, transform=_tf(False))

    class TrainSet(torch.utils.data.Dataset):
        def __len__(self):  return len(train_pos)
        def __getitem__(self, i):
            pos = train_pos[i]
            x, y = aug[idx[pos]]
            return x, y, logits[pos]

    train_loader = DataLoader(TrainSet(), 48, shuffle=True, num_workers=0)
    val_loader = DataLoader(Subset(plain, [idx[p] for p in val_pos]), 48, num_workers=0)
    test_loader = L["test"]

    runs = {}
    model, _, val_acc = train_student(train_loader, val_loader, distil=False, label="baseline")
    runs["baseline"] = {"alpha": None, "val_acc": val_acc, "test_acc": accuracy(model, test_loader)}
    for a in alphas:
        model, _, val_acc = train_student(train_loader, val_loader, distil=True,
                                          alpha=a, label=f"alpha{a}")
        runs[f"alpha{a}"] = {"alpha": a, "val_acc": val_acc, "test_acc": accuracy(model, test_loader)}

    kd = {k: v for k, v in runs.items() if v["alpha"] is not None}
    best = max(kd, key=lambda k: kd[k]["val_acc"])        # selection: validation only
    return runs, best

# runs, best = sweep()   # ~80 minutes on CPU

## 6. Results

Loaded from `results.json`, which is what the runs above wrote. Nothing below is typed in
by hand.

In [ ]:
import json

M = json.load(open("results.json"))
H, E1, E2, E3 = (M["headline"], M["experiment_1_weak_teacher"],
                 M["experiment_2_proper_teacher"], M["experiment_3_alpha_sweep"])

for name, E in [("experiment 1 — re-stemmed teacher", E1),
                ("experiment 2 — pretrained stem, 64x64", E2)]:
    print(f"{name}")
    print(f"   teacher    {E['teacher_acc']*100:6.2f}%")
    print(f"   baseline   {E['baseline_acc']*100:6.2f}%")
    print(f"   distilled  {E['distilled_acc']*100:6.2f}%   ({E['gain']*100:+.2f} points)\n")

print("experiment 3 — alpha sweep (selection on validation)")
print(f"   {'run':<12}{'alpha':>7}{'val':>9}{'test':>9}")
for k, v in E3["runs"].items():
    a = "--" if v["alpha"] is None else f"{v['alpha']:.1f}"
    print(f"   {k:<12}{a:>7}{v['val_acc']*100:>8.2f}%{v['test_acc']*100:>8.2f}%")
print(f"   selected alpha={E3['selected_alpha']} -> test {E3['selected_test_acc']*100:.2f}% "
      f"vs baseline {E3['baseline_test_acc']*100:.2f}%  ({E3['gain']*100:+.2f} points)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.9))

# (a) the two teachers, side by side
base = [E1["baseline_acc"]*100, E2["baseline_acc"]*100]
dist = [E1["distilled_acc"]*100, E2["distilled_acc"]*100]
x, w = np.arange(2), 0.34
ax[0].bar(x - w/2, base, w, color="#64748b", label="student, hard labels")
ax[0].bar(x + w/2, dist, w, color="#e11d48", label="student, distilled")
for i, (bb, dd) in enumerate(zip(base, dist)):
    ax[0].annotate(f"{dd-bb:+.1f}", (i, max(bb, dd) + 0.5), ha="center",
                   color="#b91c1c" if dd < bb else "#15803d", fontsize=9)
ax[0].set_xticks(x)
ax[0].set_xticklabels([f"weak teacher\n({E1['teacher_acc']*100:.1f}%)",
                       f"proper teacher\n({E2['teacher_acc']*100:.1f}%)"])
ax[0].set_ylim(70, 90); ax[0].set_ylabel("test accuracy (%)")
ax[0].set_title("The teacher's margin decides the sign"); ax[0].legend(frameon=False, fontsize=8)

# (b) learning curves for the proper teacher
ep = [h["epoch"] for h in E2["baseline_history"]]
ax[1].plot(ep, [h["test_acc"]*100 for h in E2["baseline_history"]], "o-", ms=3,
           color="#64748b", label="baseline")
ax[1].plot(ep, [h["test_acc"]*100 for h in E2["distilled_history"]], "o-", ms=3,
           color="#e11d48", label="distilled")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy (%)")
ax[1].set_title("The two runs sit on top of each other"); ax[1].legend(frameon=False, fontsize=8)

# (c) the sweep
alphas = [v["alpha"] for v in E3["runs"].values() if v["alpha"] is not None]
ax[2].plot(alphas, [E3["runs"][f"alpha{a}"]["val_acc"]*100 for a in alphas], "o-",
           color="#f59e0b", label="validation (selects)")
ax[2].plot(alphas, [E3["runs"][f"alpha{a}"]["test_acc"]*100 for a in alphas], "o-",
           color="#e11d48", label="test (reported)")
ax[2].axhline(E3["baseline_test_acc"]*100, ls="--", lw=1, color="#64748b", label="baseline")
ax[2].set_xlabel("alpha"); ax[2].set_ylabel("accuracy (%)")
ax[2].set_title("Sweeping alpha moves nothing"); ax[2].legend(frameon=False, fontsize=8)

for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

The middle panel is worth dwelling on. The distilled run is not merely equal at the end —
it tracks the baseline the whole way. This is not a late-schedule artifact or a bad final
epoch.

One trap to avoid: comparing the two runs' **training losses**. They are different
objectives and their absolute values say nothing about which model is better. Only accuracy
answers that.

## 7. So what is actually wrong

Two things in this setup are known to break distillation, and both are true here.

**The student cannot represent the target.** The teacher's distribution over 10 classes
encodes structure a three-block CNN with a 128-wide penultimate layer has no capacity to
reproduce. Past some point the KL term asks for something unreachable — and weighting it
harder (α = 0.9, the one run that came out *worse*) just spends more of a fixed budget on it.

**The teacher never saw what the student sees.** The cached logits were computed on the
clean image; the student trains on a random crop and a 50% horizontal flip. Roughly half
the time it is told to match the teacher's answer for a picture it is not looking at. Beyer
et al. (2022) make exactly this the central point — distillation works when teacher and
student see *identical* views, and it needs long schedules to pay off. This implementation
satisfies neither, and that is the most likely reason the gain is flat.

The conclusion is not "distillation does not work". It is that **a teacher with a real
margin is necessary but not sufficient**: experiment 1 shows what happens without the
margin (-3.0 points), and experiments 2 and 3 show the margin alone buys nothing.

### What would be tried next, in order

1. **Consistent teaching** — run the teacher live on the same augmented view instead of
   caching clean logits. Costs a teacher forward pass per batch; the most likely fix.
2. **Longer schedules** — the distilled runs were still improving at epoch 20 while the
   baseline had plateaued.
3. **More student capacity** — a wider penultimate layer, so there is something to
   distil into.

## References

* Hinton, G., Vinyals, O., Dean, J. (2015). *Distilling the Knowledge in a Neural Network.* [arXiv:1503.02531](https://arxiv.org/abs/1503.02531)
* Beyer, L., Zhai, X., Royer, A., Markeeva, L., Anil, R., Kolesnikov, A. (2022). *Knowledge Distillation: A Good Teacher is Patient and Consistent.* [arXiv:2106.05237](https://arxiv.org/abs/2106.05237)
* He, K., Zhang, X., Ren, S., Sun, J. (2016). *Deep Residual Learning for Image Recognition.* [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)